# 622. Design Circular Queue

**Difficulty:** Medium &nbsp;|&nbsp; **Topics:** array, linked-list, design, queue
&nbsp;|&nbsp; [LeetCode](https://leetcode.com/problems/design-circular-queue/)

Design your own **circular queue**: a FIFO queue of fixed capacity in which the
last position is connected back to the first, making a circle. It is also called
a **ring buffer**.

The benefit is the space *in front* of the queue. In a plain queue, once you have
walked the front pointer forward, the slots it left behind are dead - the queue
reports itself full while part of it stands empty. A circular queue wraps around
and uses them again.

Implement the `MyCircularQueue` class:

- `MyCircularQueue(k)` initializes the queue with capacity `k`.
- `enQueue(value)` inserts an element at the back. Returns `True` on success,
  `False` if the queue is full.
- `deQueue()` removes the element at the front. Returns `True` on success,
  `False` if the queue is empty.
- `Front()` returns the front element, or `-1` if the queue is empty.
- `Rear()` returns the last element, or `-1` if the queue is empty.
- `isEmpty()` / `isFull()` - what they say.

You must solve it **without using the built-in queue** of your language.

---

### Example 1

```
Input:  ["MyCircularQueue","enQueue","enQueue","enQueue","enQueue","Rear","isFull","deQueue","enQueue","Rear"]
        [[3],              [1],      [2],      [3],      [4],      [],    [],      [],       [4],      []]
Output: [null,             true,     true,     true,     false,    3,     true,    true,     true,     4]

MyCircularQueue q = new MyCircularQueue(3);
q.enQueue(1);   // true
q.enQueue(2);   // true
q.enQueue(3);   // true
q.enQueue(4);   // false - full
q.Rear();       // 3
q.isFull();     // true
q.deQueue();    // true
q.enQueue(4);   // true  - and it lands in the slot the front just left
q.Rear();       // 4
```

---

### Constraints

- `1 <= k <= 1000`
- `0 <= value <= 1000`
- At most `3000` calls will be made to `enQueue`, `deQueue`, `Front`, `Rear`,
  `isEmpty` and `isFull`.

Note `0 <= value`: **`0` is a legal value.** Anywhere you write `if self.front:`
or `return x or -1`, a stored `0` will be read as "nothing there". The test cell
has a case for exactly that.


## Before you write anything

Fourth design problem, and the first one where the expected output is a fixed
list again: #382 could return any of `n` values, #379 any free number - here every
one of the seven methods is fully determined, so the harness can go back to
comparing against a written-down answer.

You have built a queue before: #104's BFS used one you wrote as a linked list
with `first`/`last` pointers and an `O(1)` size. That one grew as needed. This one
**may not grow** - `k` slots, forever - and the whole problem lives in that
constraint. Paper first.

**1.** Take an array of 3 slots and two indices, `head` (where the front element
is) and `tail` (where the next one will go). Draw the array after each step:

```
enQueue(1)   enQueue(2)   enQueue(3)   deQueue()   enQueue(4)
```

Where did the `4` physically land? Write down the one line of arithmetic that
sent it there instead of past the end of the array.

**2.** Now the question the whole problem is about. In your drawing, find the
state after three `enQueue`s (full) and the state at the very start (empty). What
are `head` and `tail` in each?

They are the same. `head == tail` means both **empty** and **full**, and no
cleverness with two indices can separate them - the information is not there.
So you must store one more thing. Name **three** different things you could store
that would break the tie, then pick one and, for each of the seven methods, say
which piece of state it reads.

**3.** Index arithmetic, in terms of `head`, `count` and `k`:

- where does the next `enQueue` write?
- what index holds the last element (what `Rear` must read)?

Write both as one expression each. Then, in a scratch cell, evaluate `-1 % 3` in
Python. Does Python's answer save you here, and what would the same expression do
in C or Java? (This is one of the few places where Python's `%` quietly hands you
a correct program.)

**4.** Fence-posts, `k = 1`. Trace `enQueue(5)`, `isFull()`, `Front()`, `Rear()`,
`enQueue(6)`, `deQueue()`, `isEmpty()`, `Front()`. A queue of one is where an
"off by one somewhere in the middle" turns into a visible wrong answer.

**5.** `Front()` and `Rear()` return `-1` on an empty queue, which works only
because the constraints promise `0 <= value`. Same trick as #379's `get()`.
What would you return instead if values could be any integer? (There are two
respectable answers, and "raise an exception" is one of them.)

**6.** "Without using the built-in queue" rules out `collections.deque`. But a
plain Python list with `append` and `pop(0)` would *pass the tests* - so why is it
the wrong answer? What is `pop(0)` on a list of `k` elements, really? Two
notebooks in this repo already bit you with this exact cost hidden inside an
innocent-looking call - name them. Then say what the fixed array plus the modulo
buys you that `pop(0)` never can.

**7.** The **linked-list** tag has two different shapes here:

- **(a)** a singly linked list with `head` and `tail` pointers, allocating a node
  on every `enQueue` and dropping one on every `deQueue`;
- **(b)** a **circular** linked list of exactly `k` nodes, built once in
  `__init__`, with pointers walking around the ring and **no allocation ever
  again**.

Which one is the "ring buffer" the problem is named after? And in a program that
must not allocate memory while it runs - audio, embedded, a kernel log - what does
(b) buy that (a) cannot?


## Two routes - and they are the same algorithm twice

**A - fixed list, head index, count** *(write this first)*
`self.q = [0] * k`, `self.head = 0`, `self.count = 0`. `enQueue` writes at
`(head + count) % k` and bumps `count`; `deQueue` moves `head` to
`(head + 1) % k` and drops `count`; `Front` reads `q[head]`, `Rear` reads
`q[(head + count - 1) % k]`; `isEmpty` / `isFull` compare `count` to `0` and `k`.
Seven methods, all `O(1)`, `O(k)` space allocated once. This is the version to
submit.

Why `count` instead of a `tail` index: `tail` alone is the ambiguity from
question 2, so you would need a second field anyway. `count` *is* that second
field, and it happens to make `isEmpty`, `isFull` and `Rear` trivial. If you keep
`head` and `tail` instead, keep the flag or the wasted slot too - and write down
which of the three you chose and why, because that choice is the answer to this
problem.

**B - a ring of `k` nodes** *(the linked-list answer, question 7b)*
In `__init__`, build `k` nodes and close the chain: the last one's `.next` is the
first. Keep `self.front`, `self.rear` and `self.count`. `enQueue` on an empty
queue writes into `front` and points `rear` at it; otherwise it moves
`rear = rear.next`, writes there, bumps `count`. `deQueue` moves
`front = front.next` and drops `count`. Nothing is ever allocated or freed after
construction.

Now put the two next to each other:

```
route A:  i = (i + 1) % k
route B:  node = node.next
```

**That is the same operation.** `% k` is how you walk a ring you laid out in an
array; `.next` is how you walk a ring you laid out in pointers. Route B is worth
building once just to see that the modulo *was* the circular list all along.

Write A, run the tests, then write B and run the same tests - `run` only knows
the seven methods.

> One thing the harness genuinely cannot catch: a `pop(0)`-based solution passes
> every test below, because at `k = 1000` and 3 000 calls the moved bytes are too
> few to notice. Question 6 is the only thing standing between you and that
> answer. Grep your own file for `pop(0)` and `insert(0,` before you call it done.


In [ ]:
class Node :
    def __init__(self, val, next) :
        self.val = val
        self.next = next

class MyCircularQueue:
    def __init__(self, k: int):
        self.head=None
        self.last=None
        self.state:bool=True
        self.size=k
        self.currentSize = 0

    def enQueue(self, value: int) -> bool:
        self.state = self.currentSize < self.size
        if self.state:
            i = Node(value, None)
            if self.head is None :
                self.head = i
                self.last = i
            else :
                i.next = self.head
                self.head = i
            self.currentSize += 1
            return self.state
        else : return self.state
    def deQueue(self) -> bool:
        self.state = self.currentSize > 0
        if self.state:
            self.currentSize -= 1
            if self.head.next == self.last:
                self.head = None
                self.last = None
            else :
                self.head = self.head.next
            return self.state
        else : return self.state
    def Front(self) -> int:
        if self.isEmpty() :
            return -1
        return self.head.val
    def Rear(self) -> int:
        if self.isEmpty() :
            return -1
        return self.last.val
    def isEmpty(self) -> bool:
        if self.head == None : return True
        return False
    def isFull(self) -> bool:
        if self.size == self.currentSize : return True
        return False


### The test harness

Two layers. `run` is the LeetCode-style replay from #155: a list of operation
names, a list of argument lists, compared against a written-down expected output.

`stress` adds what a fixed expected list cannot cover - thousands of operations in
an order nobody chose by hand. It replays random calls against **both** your class
and `ModelQueue`, an obviously-correct queue built on `append` / `pop(0)`, and
stops at the first answer that differs. An oracle in a test is allowed to be slow;
it only has to be right. (That is also why `pop(0)` appears here and must not
appear in your solution - question 6.)

The four boolean methods are checked for being actual `bool`s, which is what
catches the most common slip of all: a method that computes the right thing and
forgets to `return` it.

Run this cell; don't edit it.


In [ ]:
import random

BOOL_METHODS = ("enQueue", "deQueue", "isEmpty", "isFull")


def run(ops, args):
    """Replay LeetCode-style (ops, args) against MyCircularQueue, collect outputs."""
    out, q = [], None
    for op, a in zip(ops, args):
        if op == "MyCircularQueue":
            q = MyCircularQueue(*a)
            out.append(None)
        else:
            out.append(getattr(q, op)(*a))
    return out


class ModelQueue:
    """The obvious slow queue - a TEST ORACLE, not an answer to the problem."""

    def __init__(self, k):
        self.k, self.items = k, []

    def enQueue(self, value):
        if len(self.items) == self.k:
            return False
        self.items.append(value)
        return True

    def deQueue(self):
        if not self.items:
            return False
        self.items.pop(0)          # O(k) - the reason this is only an oracle
        return True

    def Front(self):
        return self.items[0] if self.items else -1

    def Rear(self):
        return self.items[-1] if self.items else -1

    def isEmpty(self):
        return len(self.items) == 0

    def isFull(self):
        return len(self.items) == self.k


def stress(k, calls, seed=0):
    """Random calls replayed against both your class and the oracle."""
    random.seed(seed)
    q, model, log = MyCircularQueue(k), ModelQueue(k), []
    for _ in range(calls):
        op = random.choice(["enQueue", "enQueue", "deQueue",
                            "Front", "Rear", "isEmpty", "isFull"])
        a = [random.randint(0, 1000)] if op == "enQueue" else []
        got = getattr(q, op)(*a)
        want = getattr(model, op)(*a)
        log.append(f"{op}({a[0] if a else ''}) -> {got!r}   oracle: {want!r}")
        if op in BOOL_METHODS and not isinstance(got, bool):
            log.append(f"   !! {op} must return a bool, got {type(got).__name__}")
            return False, log
        if got != want:
            log.append(f"   !! {op} must return {want!r}, got {got!r}")
            return False, log
    return True, log


def report(name, ok, log, tail=5):
    print(f"{'OK  ' if ok else 'FAIL'} {name}")
    if not ok:
        for line in log[-tail:]:
            print(f"       {line}")


In [ ]:
# tests
TESTS = [
    ("the LeetCode example",
     ["MyCircularQueue","enQueue","enQueue","enQueue","enQueue","Rear","isFull","deQueue","enQueue","Rear"],
     [[3],[1],[2],[3],[4],[],[],[],[4],[]],
     [None,True,True,True,False,3,True,True,True,4]),

    ("k = 1 - every fence-post at once",
     ["MyCircularQueue","isEmpty","enQueue","isFull","isEmpty","Front","Rear","enQueue","deQueue","isEmpty","Front","deQueue"],
     [[1],[],[5],[],[],[],[],[6],[],[],[],[]],
     [None,True,True,True,False,5,5,False,True,True,-1,False]),

    ("everything asked of an empty queue",
     ["MyCircularQueue","isEmpty","isFull","Front","Rear","deQueue"],
     [[2],[],[],[],[],[]],
     [None,True,False,-1,-1,False]),

    ("the wrap: two slots freed at the front get reused",
     ["MyCircularQueue","enQueue","enQueue","enQueue","deQueue","deQueue","enQueue","enQueue","Front","Rear","isFull","enQueue"],
     [[3],[1],[2],[3],[],[],[4],[5],[],[],[],[6]],
     [None,True,True,True,True,True,True,True,3,5,True,False]),

    ("question 2: head == tail, drained after being full - empty, NOT full",
     ["MyCircularQueue","enQueue","enQueue","isFull","deQueue","deQueue","isEmpty","isFull","Front","Rear","enQueue","Front","Rear"],
     [[2],[1],[2],[],[],[],[],[],[],[],[9],[],[]],
     [None,True,True,True,True,True,True,False,-1,-1,True,9,9]),

    ("deQueue past empty must not corrupt anything",
     ["MyCircularQueue","deQueue","deQueue","enQueue","Front","Rear","isEmpty","deQueue","isEmpty"],
     [[2],[],[],[7],[],[],[],[],[]],
     [None,False,False,True,7,7,False,True,True]),

    ("0 is a legal value - any truthiness check dies here",
     ["MyCircularQueue","enQueue","Front","Rear","isEmpty","enQueue","isFull","deQueue","Front"],
     [[2],[0],[],[],[],[0],[],[],[]],
     [None,True,0,0,False,True,True,True,0]),

    ("five laps around a 3-slot ring",
     ["MyCircularQueue","enQueue","enQueue","enQueue",
      "deQueue","enQueue","deQueue","enQueue","deQueue","enQueue",
      "deQueue","enQueue","deQueue","enQueue","Front","Rear","isFull"],
     [[3],[1],[2],[3],
      [],[4],[],[5],[],[6],
      [],[7],[],[8],[],[],[]],
     [None,True,True,True,
      True,True,True,True,True,True,
      True,True,True,True,6,8,True]),
]

for name, ops, args, expected in TESTS:
    got = run(ops, args)
    report(name, got == expected, [f"got  {got}", f"want {expected}"], tail=2)

# random sequences against the oracle - the last one is the constraint ceiling
for k, calls, seed in [(1,200,1),(2,400,2),(3,600,3),(7,2000,4),(1000,3000,5)]:
    report(f"stress: k={k}, {calls} random calls (seed {seed})", *stress(k, calls, seed))

# see it, do not just trust the pass/fail
print("\ntrace of the LeetCode example:")
ops, args, expected = TESTS[0][1], TESTS[0][2], TESTS[0][3]
for op, a, got, want in zip(ops, args, run(ops, args), expected):
    call = f"{op}({', '.join(map(str, a))})"
    print(f"  {call:24} -> {got!r:6} {'' if got == want else f'   want {want!r}'}")


## After it passes

- **Write down your answer to question 2.** Which of the three tie-breakers did
  you keep - a `count`, one deliberately wasted slot, or a boolean flag? Then
  cost the other two. For the wasted slot: you allocate `k + 1` cells and
  `isFull` becomes `(tail + 1) % (k + 1) == head` - check for yourself that this
  still stores exactly `k` items, and say which method got *simpler* in exchange
  for the extra cell. For the flag: name the two methods that must flip it, and
  the wrong answer you get from forgetting one of them.
- **Build route B and re-run the same tests.** Then say the sentence out loud:
  `(i + 1) % k` and `node.next` are the same operation. Which of the two would
  you rather debug, and which one would you rather write in C?
- **Compare with `deque(maxlen=3)`.** Push four items into one in a scratch cell.
  It does not refuse the fourth - it *drops the oldest*. Same structure, opposite
  policy for a full buffer. Which policy does a log want? A task queue? An audio
  buffer? "What happens when it is full" is a product decision, not a data
  structure one.
- **Where this actually lives.** Ring buffers are how audio drivers, `dmesg`,
  network cards and single-producer/single-consumer pipes work, precisely because
  the memory is fixed at construction and nothing allocates while the program
  runs. Look at your `__init__` and your `enQueue` again: that property is the
  whole point, and route A has it too.
- **#379 was the same job with a different bookkeeper.** Both hand out slots from
  a fixed pool. There, "which slots are free" was a `set` and order did not matter;
  here, order is everything and two integers do the whole job. What made the
  cheaper state enough?
- **The `O(n)`-hidden-in-a-friendly-call family**, one line each on where it bit
  you: #103's `insert(0, x)`, #105's `.index()`, #155's `min(...)`, and this
  problem's `pop(0)`. Any test suite will pass all four. That is the point.
- Siblings: **#641 Design Circular Deque** - this class plus `insertFront` and
  `deleteLast`, roughly fifteen minutes once route A works, and the two new
  methods are where a `count` really earns its keep; #232 Implement Queue using
  Stacks and #225 the reverse, both about paying somewhere else to keep a promise;
  #146 LRU Cache when you want the real design problem.


In [ ]:
class MyCircularQueue:

    def __init__(self, k: int):
        self.queue = [0] * k
        self.size = k
        self.currentSize = 0
        self.front = 0
        self.rear = 0
        self.state = True

    def enQueue(self, value: int) -> bool:
        self.state = not self.isFull()

        if self.state:
            self.queue[self.rear] = value
            self.rear = (self.rear + 1) % self.size
            self.currentSize += 1

        return self.state

    def deQueue(self) -> bool:
        self.state = not self.isEmpty()

        if self.state:
            self.front = (self.front + 1) % self.size
            self.currentSize -= 1

        return self.state

    def Front(self) -> int:
        if self.isEmpty():
            return -1
        return self.queue[self.front]

    def Rear(self) -> int:
        if self.isEmpty():
            return -1
        return self.queue[(self.rear - 1) % self.size]

    def isEmpty(self) -> bool:
        return self.currentSize == 0

    def isFull(self) -> bool:
        return self.currentSize == self.size